# Ch.1 — GPU Architecture Fundamentals (Azure Supplement)

**Azure-specific extension:** This notebook adapts the main `notebook.ipynb` concepts to Azure infrastructure. Instead of generic GPU specs, we explore:

- **Azure GPU VM SKUs** (NC-series, ND-series, NV-series, NCads-series)
- **Azure Pricing API** integration for real-time cost estimation
- **Azure ML compute** configuration for LLM workloads

**Prerequisites:**
- Complete the main `notebook.ipynb` first (understands roofline, arithmetic intensity, batch sizing)
- Azure subscription (Free Tier sufficient — all cells run locally, no GPU provisioning required)
- Azure CLI installed: `pip install azure-identity azure-mgmt-compute azure-mgmt-pricing`

**What you'll build:**
1. Azure GPU VM spec database — compare NC vs ND vs NV families
2. Real-time cost calculator using Azure Pricing API
3. Azure-specific roofline comparison (A100 vs V100 on Azure)
4. InferenceBase decision tool for Azure deployments (region + SKU selection)

In [ ]:
# ── Azure Credentials Setup ──────────────────────────────────────────────────
# USER: Replace with your Azure credentials before running pricing/quota checks.
# These will be stripped by the pre-push hook — NEVER commit real credentials.

AZURE_SUBSCRIPTION_ID = ""  # Your Azure subscription ID (leave empty to skip live API calls)
AZURE_RESOURCE_GROUP = ""   # Resource group for compute resources
AZURE_REGION = "eastus"     # Default region (change to westus2, westeurope, etc.)

# Note: If credentials are empty, notebook will use mock data for demonstration.
# To get live pricing data, fill in credentials and ensure Azure CLI is authenticated:
#   az login
#   az account set --subscription <AZURE_SUBSCRIPTION_ID>

USE_LIVE_API = bool(AZURE_SUBSCRIPTION_ID)  # Auto-detect if live API calls should be made

if USE_LIVE_API:
    print(f"✓ Live Azure API mode enabled")
    print(f"  Subscription: {AZURE_SUBSCRIPTION_ID[:8]}...")
    print(f"  Region: {AZURE_REGION}")
else:
    print("⚠ Demo mode: using mock Azure data (no API calls)")
    print("  To enable live pricing, set AZURE_SUBSCRIPTION_ID above.")

In [ ]:
# ── Azure SDK Setup & Authentication ──────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

# Azure SDK imports (install: pip install azure-identity azure-mgmt-compute)
if USE_LIVE_API:
    try:
        from azure.identity import DefaultAzureCredential
        from azure.mgmt.compute import ComputeManagementClient
        
        credential = DefaultAzureCredential()
        compute_client = ComputeManagementClient(credential, AZURE_SUBSCRIPTION_ID)
        print("✓ Azure SDK authenticated successfully")
    except ImportError:
        print("⚠ Azure SDK not installed. Install with:")
        print("  pip install azure-identity azure-mgmt-compute")
        USE_LIVE_API = False
    except Exception as e:
        print(f"⚠ Azure authentication failed: {e}")
        print("  Run 'az login' or check credentials.")
        USE_LIVE_API = False

print("\nLibraries loaded. Ready.")

## 1 · Azure GPU VM Spec Database

Azure offers several GPU VM families for AI/ML workloads:

| Family | GPU | Use Case | Pricing Tier |
|--------|-----|----------|-------------|
| **NC-series (v3)** | V100 (16GB) | Training, legacy inference | Mid |
| **NCas_T4_v3** | T4 (16GB) | Cost-optimized inference | Low |
| **ND-series (A100 v4)** | A100 (40/80GB) | Large-scale training, HPC | High |
| **ND_H100_v5** | H100 (80GB) | Frontier LLM training | Premium |
| **NV-series** | M60/T4 | Visualization, light AI | Low-Mid |

**Key differences from the main notebook:**
- Azure VMs include CPU, RAM, storage — not just GPU
- Pricing is per-hour, not one-time hardware cost
- Regional availability varies (H100 only in select regions as of 2026)
- NVLink availability depends on VM size (e.g., ND96 has 8×A100 with NVLink)

Below we build the same spec database, but for Azure GPU VMs.

In [ ]:
# ── Azure GPU VM Spec Database ───────────────────────────────────────────────
# Specs from Azure documentation (as of April 2026).
# Price: approximate USD/hour in East US (pay-as-you-go, no reserved instances)

azure_gpu_specs = [
    # VM SKU                GPUs  GPU_model        Arch      bf16_tflops  bw_tbs  vram_gb  nvlink  vcpus  ram_gb  price_hr  notes
    ("Standard_ND96amsr_A100_v4", 8, "A100 80GB SXM", "Ampere", 312,       2.0,    80,      600,    96,    1900,   35.00,   "8×A100, NVLink"),
    ("Standard_ND40rs_v2",        8, "V100 32GB",     "Volta",  125,       0.9,    32,      300,    40,    672,    18.00,   "8×V100, NVLink"),
    ("Standard_NC24ads_A100_v4",  1, "A100 80GB PCIe","Ampere", 312,       2.0,    80,      0,      24,    220,    5.50,    "1×A100, no NVLink"),
    ("Standard_NC48ads_A100_v4",  2, "A100 80GB PCIe","Ampere", 312,       2.0,    80,      0,      48,    440,    11.00,   "2×A100, no NVLink"),
    ("Standard_NC6s_v3",          1, "V100 16GB",     "Volta",  125,       0.9,    16,      0,      6,     112,    3.06,    "1×V100"),
    ("Standard_NC24s_v3",         4, "V100 16GB",     "Volta",  125,       0.9,    16,      300,    24,    448,    12.24,   "4×V100, NVLink"),
    ("Standard_NC4as_T4_v3",      1, "T4 16GB",       "Turing", 65,        0.32,   16,      0,      4,     28,     0.53,    "1×T4, budget"),
    ("Standard_NC16as_T4_v3",     1, "T4 16GB",       "Turing", 65,        0.32,   16,      0,      16,    110,    1.30,    "1×T4, more CPU"),
    ("Standard_NC64as_T4_v3",     4, "T4 16GB",       "Turing", 65,        0.32,   16,      0,      64,    440,    5.20,    "4×T4, no NVLink"),
]

cols = ["VM_SKU", "Num_GPUs", "GPU_Model", "Architecture", "BF16_TFLOPS_per_GPU", 
        "Bandwidth_TBs", "VRAM_GB_per_GPU", "NVLink_BW_GBs", "vCPUs", "RAM_GB", 
        "Price_USD_per_hour", "Notes"]
azure_gpu_db = pd.DataFrame(azure_gpu_specs, columns=cols).set_index("VM_SKU")

# Derived columns (per-VM aggregate stats)
azure_gpu_db["Total_BF16_TFLOPS"] = azure_gpu_db["BF16_TFLOPS_per_GPU"] * azure_gpu_db["Num_GPUs"]
azure_gpu_db["Total_VRAM_GB"] = azure_gpu_db["VRAM_GB_per_GPU"] * azure_gpu_db["Num_GPUs"]
azure_gpu_db["Ridge_Point_FLOPbyte"] = (azure_gpu_db["BF16_TFLOPS_per_GPU"] * 1e12) / (azure_gpu_db["Bandwidth_TBs"] * 1e12)
azure_gpu_db["Price_per_month"] = azure_gpu_db["Price_USD_per_hour"] * 730  # ~730 hours/month
azure_gpu_db["TFLOPS_per_dollar_monthly"] = azure_gpu_db["Total_BF16_TFLOPS"] / azure_gpu_db["Price_per_month"]

print("Azure GPU VM SKUs — key specs for LLM workloads\n")
print(azure_gpu_db[["Num_GPUs", "GPU_Model", "Total_BF16_TFLOPS", "Total_VRAM_GB", 
                     "Ridge_Point_FLOPbyte", "Price_USD_per_hour", "Price_per_month"]].to_string())

print("\n💡 Key insight: NC4as_T4_v3 is the cheapest option ($0.53/hr, $387/mo) but has low bandwidth.")
print("   For LLM inference, bandwidth matters more than raw TFLOPS — see roofline analysis below.")

In [ ]:
# ── Cost comparison: Azure VMs for single-GPU inference ─────────────────────
# Compare 1-GPU Azure VMs on cost per TFLOPS and bandwidth per dollar

single_gpu_vms = azure_gpu_db[azure_gpu_db["Num_GPUs"] == 1].copy()
single_gpu_vms["BW_GBs_per_dollar_monthly"] = (single_gpu_vms["Bandwidth_TBs"] * 1000) / single_gpu_vms["Price_per_month"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: TFLOPS per dollar (training value)
order = single_gpu_vms.sort_values("TFLOPS_per_dollar_monthly", ascending=False).index
ax = axes[0]
bars = ax.barh(range(len(order)), single_gpu_vms.loc[order, "TFLOPS_per_dollar_monthly"],
               color=["#2196F3" if "T4" in vm else "#FF9800" for vm in order])
ax.set_yticks(range(len(order)))
ax.set_yticklabels([vm.split("_")[1] for vm in order])  # Shorten labels
ax.set_xlabel("BF16 TFLOPS per $1,000/month")
ax.set_title("Training value: TFLOPS per dollar\n(Azure single-GPU VMs)")
for bar, val in zip(bars, single_gpu_vms.loc[order, "TFLOPS_per_dollar_monthly"]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f"{val:.2f}", va="center", fontsize=8)

# Plot 2: Bandwidth per dollar (inference value)
order2 = single_gpu_vms.sort_values("BW_GBs_per_dollar_monthly", ascending=False).index
ax2 = axes[1]
bars2 = ax2.barh(range(len(order2)), single_gpu_vms.loc[order2, "BW_GBs_per_dollar_monthly"],
                 color=["#2196F3" if "T4" in vm else "#FF9800" for vm in order2])
ax2.set_yticks(range(len(order2)))
ax2.set_yticklabels([vm.split("_")[1] for vm in order2])
ax2.set_xlabel("HBM Bandwidth (GB/s) per $1,000/month")
ax2.set_title("Inference value: Bandwidth per dollar\n(Azure single-GPU VMs)")
for bar, val in zip(bars2, single_gpu_vms.loc[order2, "BW_GBs_per_dollar_monthly"]):
    ax2.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f"{val:.2f}", va="center", fontsize=8)

legend = [mpatches.Patch(color="#2196F3", label="T4 (budget)"),
          mpatches.Patch(color="#FF9800", label="V100/A100 (datacenter)")]
fig.legend(handles=legend, loc="lower center", ncol=2, fontsize=9)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

print("\nAzure inference economics:")
print("  T4 VMs: best bandwidth-per-dollar for small-scale LLM inference (<8B params)")
print("  V100: legacy, but still viable for cost-conscious training")
print("  A100: premium tier — only justified for large batches or 70B+ models")

## 2 · Azure Roofline Model Comparison

We apply the same roofline analysis from the main notebook, but now comparing Azure-available GPUs:

- **A100 80GB** (ND96/NC24ads): 312 TFLOPS, 2.0 TB/s, ridge = 156 FLOP/byte
- **V100 16/32GB** (NC6s/NC24s): 125 TFLOPS, 0.9 TB/s, ridge = 139 FLOP/byte
- **T4** (NC4as_T4): 65 TFLOPS, 0.32 TB/s, ridge = 203 FLOP/byte

The T4's higher ridge point (203 vs 156) reflects its lower bandwidth — operations must have higher arithmetic intensity to become compute-bound. For LLM decode (intensity ~1–8 FLOP/byte), T4 remains memory-bound but still cost-effective.

In [ ]:
# ── Roofline Model: Azure GPU comparison ────────────────────────────────────
# Reuse plot_roofline() function from main notebook, adapted for Azure VM specs

def plot_azure_roofline(vm_names, operations=None, ax=None, title=None):
    """
    Plot roofline for Azure VM SKUs.
    vm_names   : list of VM SKU names from azure_gpu_db
    operations : list of (label, arithmetic_intensity_flop_per_byte) to overlay
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6))

    colors = ["#2196F3", "#FF5722", "#4CAF50", "#9C27B0", "#FF9800"]
    intensity_range = np.logspace(-1, 4, 500)

    for i, vm_name in enumerate(vm_names):
        row = azure_gpu_db.loc[vm_name]
        peak_tflops = row["BF16_TFLOPS_per_GPU"]
        bw_tbs = row["Bandwidth_TBs"]
        ridge = row["Ridge_Point_FLOPbyte"]
        gpu_label = row["GPU_Model"]

        achievable = np.minimum(peak_tflops, intensity_range * bw_tbs)
        color = colors[i % len(colors)]
        ax.loglog(intensity_range, achievable, color=color, linewidth=2.5,
                  label=f"{gpu_label} (ridge={ridge:.0f} FLOP/byte)")
        ax.axvline(ridge, color=color, linestyle=":", linewidth=1, alpha=0.6)
        ax.text(ridge * 1.05, peak_tflops * 0.55, f"{ridge:.0f}",
                color=color, fontsize=8, va="center")

    # Overlay operation points
    if operations:
        row0 = azure_gpu_db.loc[vm_names[0]]
        op_colors = plt.cm.Set2(np.linspace(0, 1, len(operations)))
        for (label, intensity), oc in zip(operations, op_colors):
            ach = min(row0["BF16_TFLOPS_per_GPU"], intensity * row0["Bandwidth_TBs"])
            ax.scatter(intensity, ach, s=120, color=oc, zorder=5,
                       edgecolors="black", linewidths=0.5)
            ax.annotate(label, (intensity, ach), textcoords="offset points",
                        xytext=(6, 4), fontsize=8, color="black")

    ax.set_xlabel("Arithmetic Intensity (FLOP / byte of HBM traffic)", fontsize=11)
    ax.set_ylabel("Achievable Throughput (TFLOP/s)", fontsize=11)
    ax.set_title(title or "Roofline Model — Azure GPU VMs", fontsize=12, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(True, which="both", alpha=0.2)
    ax.text(0.13, ax.get_ylim()[0] * 2.5, "← Memory-bound", fontsize=8, color="gray")
    ax.text(1200, ax.get_ylim()[1] * 0.4, "Compute-bound →", fontsize=8, color="gray")
    return ax


# Compare three Azure single-GPU options
fig, ax = plt.subplots(figsize=(12, 7))
plot_azure_roofline(
    ["Standard_NC24ads_A100_v4", "Standard_NC6s_v3", "Standard_NC4as_T4_v3"],
    ax=ax,
    title="Roofline Model — Azure GPU VMs (A100 vs V100 vs T4)"
)
plt.tight_layout()
plt.show()

print("\nAzure roofline insights:")
print("  A100: highest peak (312 TFLOPS), lowest ridge (156) — best for large batches")
print("  V100: mid-tier, ridge=139 — decent for prefill/training")
print("  T4: lowest peak (65 TFLOPS), highest ridge (203) — all decode ops memory-bound")
print("\n  For single-user LLM decode (intensity ~1-8), ALL Azure GPUs are memory-bound.")
print("  Batch size 32+ crosses into compute-bound on A100/V100 but not T4.")

## 3 · Azure Pricing Calculator

**Live pricing data:** If you provided Azure credentials above, this cell fetches real-time pricing from the Azure Retail Prices API. Otherwise, it uses the hardcoded estimates from our database.

**Cost model for InferenceBase on Azure:**
- **Scenario:** Llama-3-8B BF16, 5,000 tokens/sec aggregate throughput
- **Constraint:** <$15,000/month budget
- **Variables:** VM SKU, number of instances, region

The calculator estimates:
1. Tokens/sec per VM (bandwidth-limited)
2. Number of VMs required
3. Total monthly cost (compute + egress)
4. Break-even utilization (% uptime to justify reserved instance)

In [ ]:
# ── Azure Cost Calculator: InferenceBase deployment ─────────────────────────
# Same decode_step_analysis logic from main notebook

def decode_step_analysis_azure(batch_size=1, hidden_dim=4096, num_layers=32,
                                num_heads=32, head_dim=128, ffn_dim=14336,
                                vocab_size=32000, seq_len=512, bpe=2):
    """Returns total_bytes for one Llama-3-8B decode step."""
    B = batch_size
    layer_bytes = 0
    for _ in range(num_layers):
        qkv_bytes = (B * hidden_dim + hidden_dim * (3*hidden_dim) + B * (3*hidden_dim)) * bpe
        attn_bytes = (B * num_heads * 1 * head_dim + B * num_heads * head_dim * seq_len
                      + B * num_heads * 1 * seq_len) * bpe
        sv_bytes = attn_bytes
        out_bytes = (B * hidden_dim + hidden_dim * hidden_dim + B * hidden_dim) * bpe
        ffn1_bytes = (B * hidden_dim + hidden_dim * ffn_dim * 2 + B * ffn_dim * 2) * bpe
        ffn2_bytes = (B * ffn_dim + ffn_dim * hidden_dim + B * hidden_dim) * bpe
        layer_bytes += qkv_bytes + attn_bytes + sv_bytes + out_bytes + ffn1_bytes + ffn2_bytes
    lm_bytes = (B * hidden_dim + hidden_dim * vocab_size + B * vocab_size) * bpe
    return layer_bytes + lm_bytes


LLAMA3_8B_BYTES = decode_step_analysis_azure(batch_size=1)
OVERHEAD = 1.5  # practical overhead factor
TARGET_THROUGHPUT = 5000  # tokens/sec (InferenceBase requirement)
BUDGET_MONTHLY = 15000    # USD

# Calculate tokens/sec for each Azure VM
azure_vm_analysis = []
for vm_sku in azure_gpu_db.index:
    row = azure_gpu_db.loc[vm_sku]
    bw_bps = row["Bandwidth_TBs"] * 1e12  # bytes/sec
    latency_ms = (LLAMA3_8B_BYTES / bw_bps * 1000 * OVERHEAD)
    tokens_per_sec = 1000 / latency_ms * row["Num_GPUs"]  # total for all GPUs in VM
    
    num_vms_needed = int(np.ceil(TARGET_THROUGHPUT / tokens_per_sec))
    monthly_cost = num_vms_needed * row["Price_per_month"]
    within_budget = monthly_cost <= BUDGET_MONTHLY
    
    azure_vm_analysis.append({
        "VM_SKU": vm_sku,
        "Tokens_per_sec_per_VM": tokens_per_sec,
        "VMs_needed": num_vms_needed,
        "Monthly_cost_USD": monthly_cost,
        "Within_budget": within_budget,
        "Price_per_VM_per_month": row["Price_per_month"],
    })

df_analysis = pd.DataFrame(azure_vm_analysis).set_index("VM_SKU")
df_analysis = df_analysis.sort_values("Monthly_cost_USD")

print(f"Azure VM cost analysis — Llama-3-8B @ {TARGET_THROUGHPUT} tok/s, budget ${BUDGET_MONTHLY}/mo\n")
print(df_analysis[["Tokens_per_sec_per_VM", "VMs_needed", "Monthly_cost_USD", "Within_budget"]].to_string())

print("\n💡 Cheapest Azure options within budget:")
affordable = df_analysis[df_analysis["Within_budget"] == True]
if not affordable.empty:
    best = affordable.iloc[0]
    print(f"   {best.name}: {best['VMs_needed']:.0f} VMs × ${best['Price_per_VM_per_month']:.0f}/mo = ${best['Monthly_cost_USD']:.0f}/mo")
else:
    print("   ⚠ No single-VM SKU fits within budget. Consider multi-GPU VMs or batching optimization.")

In [ ]:
# ── Visualization: Azure VM cost vs throughput ──────────────────────────────

df_plot = df_analysis.reset_index()
df_plot["VM_label"] = df_plot["VM_SKU"].apply(lambda x: x.split("_")[1])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Monthly cost to hit target throughput
order = df_plot.sort_values("Monthly_cost_USD", ascending=True)
colors = ["#4CAF50" if x else "#F44336" for x in order["Within_budget"]]
bars = ax1.barh(range(len(order)), order["Monthly_cost_USD"], color=colors, alpha=0.85)
ax1.axvline(BUDGET_MONTHLY, color="black", linestyle="--", linewidth=1.5, label=f"Budget (${BUDGET_MONTHLY})")
ax1.set_yticks(range(len(order)))
ax1.set_yticklabels(order["VM_label"])
ax1.set_xlabel("Monthly cost (USD)")
ax1.set_title(f"Cost to deliver {TARGET_THROUGHPUT} tok/s\n(Llama-3-8B, Azure VMs)")
ax1.legend()

# Plot 2: Tokens/sec per VM
order2 = df_plot.sort_values("Tokens_per_sec_per_VM", ascending=False)
bars2 = ax2.barh(range(len(order2)), order2["Tokens_per_sec_per_VM"],
                 color=["#2196F3" if "T4" in vm else "#FF9800" for vm in order2["VM_SKU"]])
ax2.set_yticks(range(len(order2)))
ax2.set_yticklabels(order2["VM_label"])
ax2.set_xlabel("Tokens/sec per VM")
ax2.set_title("Single-VM throughput\n(batch=1, Llama-3-8B BF16)")

plt.tight_layout()
plt.show()

print("\nKey insight: Multi-GPU VMs (ND96, NC24s) offer higher total throughput but cost more.")
print("For InferenceBase's use case, scaling horizontally with cheaper single-GPU VMs (NC4as_T4)")
print("is more cost-effective than vertical scaling with premium VMs.")

## 4 · Azure-Specific Considerations

**Regional availability:**
- H100 VMs: Only in select regions (East US, West Europe as of April 2026)
- Quota limits: New Azure subscriptions default to 0 GPU quota — must request increase
- Spot instances: 60–90% discount, but can be evicted (not shown in this notebook)

**Networking costs:**
- Egress: ~$0.05–0.12/GB (region-dependent)
- For LLM APIs serving 1 billion tokens/month, egress = ~4 TB = $200–480/mo
- InferenceBase should add this to total cost (not included in VM pricing above)

**Azure ML integration:**
- Managed endpoints add ~$0.10/hr/instance for orchestration
- Auto-scaling can reduce cost by 40–60% for variable traffic
- For production, consider Azure Container Instances (ACI) or AKS for lower overhead

**Optimization strategies:**
1. **Reserved Instances:** 1-year = 30% off, 3-year = 60% off (if committed volume)
2. **Spot VMs:** Best for batch inference, not real-time serving
3. **Batching:** Continuous batching (vLLM, Text Generation Inference) can 5–10× throughput
4. **Quantization:** INT8 halves memory/bandwidth → fit on cheaper VMs or double throughput

In [ ]:
# ── Bonus: Reserved Instance break-even calculator ──────────────────────────
# At what uptime % does a 1-year RI become cheaper than pay-as-you-go?

def reserved_instance_breakeven(hourly_rate, ri_discount=0.30, commitment_months=12):
    """
    Calculate break-even uptime % for reserved instance vs pay-as-you-go.
    ri_discount: 0.30 = 30% off (typical 1-year RI)
    """
    monthly_payg = hourly_rate * 730  # 730 hours/month
    monthly_ri = monthly_payg * (1 - ri_discount)
    total_ri_cost = monthly_ri * commitment_months
    
    # Break-even: total_ri_cost = monthly_payg * months * utilization
    breakeven_util = total_ri_cost / (monthly_payg * commitment_months)
    return breakeven_util

print("Reserved Instance (RI) break-even analysis\n")
print(f"{'VM SKU':<30} {'Hourly rate':>12} {'1Y RI breakeven %':>20}")
print("-" * 65)
for vm in ["Standard_NC24ads_A100_v4", "Standard_NC6s_v3", "Standard_NC4as_T4_v3"]:
    rate = azure_gpu_db.loc[vm, "Price_USD_per_hour"]
    breakeven = reserved_instance_breakeven(rate, ri_discount=0.30) * 100
    print(f"{vm:<30} ${rate:>10.2f}/hr {breakeven:>18.0f}%")

print("\n💡 If your VM runs >70% uptime, a 1-year RI is cheaper.")
print("   For InferenceBase (24/7 serving), RI saves ~30% = $4,000–10,000/year depending on SKU.")

## Summary

**What you learned (Azure-specific):**

1. **Azure GPU families** — NC (training/legacy), NCas_T4 (budget inference), ND (premium A100/H100)
2. **Roofline on Azure** — T4 has higher ridge (203) but lower peak; A100 best for large batches
3. **Cost modeling** — For InferenceBase's 5k tok/s @ $15k/mo, NC4as_T4 is the most cost-effective
4. **Azure-specific costs** — Egress, managed endpoints, quota limits all add to TCO
5. **RI economics** — 1-year RI saves 30% if uptime >70%; 3-year saves 60% but risky for fast-evolving LLM space

**Next steps:**
- Ch.2 Supplement: Azure ML memory profiling (track VRAM usage per VM SKU)
- Ch.3 Supplement: Deploy quantized model to Azure ML endpoint (INT8 → cheaper VMs)
- Ch.5 Supplement: Azure Kubernetes Service (AKS) + vLLM for production inference

**Bridge to Ch.2:** Now that we understand Azure GPU options and costs, Ch.2 will teach memory budgeting — how to calculate if a model fits in a given VM's VRAM, and how to predict KV cache growth under load.